# 🏃 Personal Running Coach Agent Swarm
## Phase 4 — Agents

**Project:** Agentic AI Course — Will Sutherland, May 2026

Run the setup cell first every session, then jump to whichever agent you want.

| Agent | Status | Cell |
|---|---|---|
| Feedback Agent | ✅ Ready | Section 2 |
| Recovery Agent | 🔜 Coming | Section 3 |
| Planner Agent  | 🔜 Coming | Section 4 |
| Coordinator    | ✅ Ready | Section 5 |

---
## 1. Setup — Run This First Every Session

In [1]:
# ── Install dependencies ───────────────────────────────────────────────────────
%pip install -q chromadb google-generativeai langchain langchain-google-genai langgraph stravalib

# ── Mount Drive & set paths ────────────────────────────────────────────────────
import os, sys
from google.colab import drive, userdata

drive.mount('/content/drive', force_remount=False)

BASE_DIR = "/content/drive/MyDrive/running_coach"
sys.path.insert(0, f"{BASE_DIR}/tools")
sys.path.insert(0, f"{BASE_DIR}/agents")

# ── Load API key ───────────────────────────────────────────────────────────────
GEMINI_API_KEY = userdata.get('key')

# ── Sanity checks ─────────────────────────────────────────────────────────────
checks = {
    'GEMINI_API_KEY':       GEMINI_API_KEY,
    'workouts_normalized':  os.path.exists(f"{BASE_DIR}/data/processed/workouts_normalized.json"),
    'garmin.db':            os.path.exists(f"{BASE_DIR}/data/raw/garmin/garmin.db"),
    'chroma memory':        os.path.exists(f"{BASE_DIR}/memory/chroma"),
    'feedback_agent.py':    os.path.exists(f"{BASE_DIR}/agents/feedback_agent.py"),
}

all_ok = True
for k, v in checks.items():
    status = '✅' if v else '❌ MISSING'
    if not v:
        all_ok = False
    print(f"  {status}  {k}")

if all_ok:
    print("\n✅ All checks passed — ready to run agents.")
else:
    print("\n⚠️  Fix missing items before running agents.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  ✅  GEMINI_API_KEY
  ✅  workouts_normalized
  ✅  garmin.db
  ✅  chroma memory
  ✅  feedback_agent.py

✅ All checks passed — ready to run agents.


---
## 2. Feedback Agent

Analyses a completed workout — pace vs target, HR, training load,
recovery context, and patterns from similar past sessions.

**Options:**
- Run with no arguments → presents a menu (most recent run, most recent easy run, or specific date)
- Pass a date string → jumps straight to that workout

In [ ]:
from feedback_agent import FeedbackAgent

agent = FeedbackAgent(api_key=GEMINI_API_KEY)

# ── Choose one of these ────────────────────────────────────────────────────────
agent.run()                    # Interactive menu
# agent.run(date='2026-04-22') # Jump straight to a specific date

---
## 3. Recovery Agent

Assesses current recovery status and advises whether to proceed with
a planned session, modify it, or back off.

**Modes:**
- No arguments → interactive menu (check-in or pre-workout)
- Pass a session type → jumps straight to pre-workout check

**Session types:** `easy` / `marathon` / `threshold` / `1hr` / `fartlek` / `8k` / `vo2max`

In [ ]:
from recovery_agent import RecoveryAgent

agent = RecoveryAgent(api_key=GEMINI_API_KEY)

# ── Choose one of these ────────────────────────────────────────────────────────
agent.run()                           # Interactive menu
# agent.run(planned_session='threshold') # Pre-workout check for threshold session

---
## 4. Planner Agent

Generates next week's training plan or reviews this week's planned sessions
against current recovery signals. Applies taper logic automatically based
on days remaining to race.

**Modes:**
- No arguments → interactive menu (generate or review)
- `mode='generate'` → jumps straight to next week's plan
- `mode='review'` → asks for this week's sessions then reviews them

In [ ]:
from planner_agent import PlannerAgent

agent = PlannerAgent(api_key=GEMINI_API_KEY)

# ── Choose one of these ────────────────────────────────────────────────────────
agent.run()                    # Interactive menu
# agent.run(mode='generate')   # Jump straight to next week's plan
# agent.run(mode='review')     # Review this week's sessions

---
## 5. Coordinator Agent

The user-facing entry point for the full swarm. Accepts natural language
queries, routes to the right specialist agent(s), synthesizes multi-agent
responses, and runs a self-critique safety pass before delivery.

**Try these queries:**
- `'How did my run go today?'` → routes to Feedback
- `'Am I recovered enough for tomorrow?'` → routes to Recovery
- `'What should next week look like?'` → routes to Planner
- `'How did today\'s run go and should I do tomorrow\'s threshold?'` → Feedback + Recovery
- No query → shows the menu

In [ ]:
from coordinator_agent import CoordinatorAgent

agent = CoordinatorAgent(api_key=GEMINI_API_KEY)

# ── Choose one of these ────────────────────────────────────────────────────────
agent.run()                                                     # Interactive menu
# agent.run(query='How did my most recent run go?')             # Feedback only
# agent.run(query='Am I recovered enough for tomorrow?')        # Recovery only
# agent.run(query="How did today go and should I do tomorrow's threshold?")  # Multi

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ Coordinator Agent initialized.
   Specialist agents: Feedback | Recovery | Planner

RUNNING COACH — What would you like to know?

   Last run: 2026-05-02 | 14.01km | 4:10/km

  Just type your question, or choose from the menu:

  [1] Analyse a recent workout
  [2] Daily check-in / pre-workout readiness
  [3] Generate or review weekly training plan
  [4] Ask anything — I'll figure out which agents to call
  [Q] Quit


🧠 Classifying query...

   Routing to: ['feedback']

RUNNING COACH
   📊 Calling Feedback Agent...
  ℹ️  1 record(s) skipped (no training_load — not Garmin-enriched). Treated as 0 load.
   🔀 Synthesizing responses...
   🔍 Running self-critique pass...
   Self-critique: ✅ APPROVED

**Numbers** - Target HM pace: 3:59/km. Fast efforts (total ~6km): Avg 3:52/km (Laps 6-10) and 3:43/km (Laps 12-13). Avg HR in fast efforts: 172 bpm (mid-Zone 4). Load: 214.6, TSB: 6.8 (Fresh).

**How it went** - Will executed strong efforts above or at his sub-1:24 HM target pace (3:59/km), tota